**Специальные фильтры**




In [1]:
# Программа расчёта специального фильтра
# Данная программа позволяет преобразовать 1024 отсчёта исходной
# импульсной характеристики (ИХ) (Рис. 17.1а) в (M + 1) отсчёт
# результирующей ИХ (Рис. 17.1в), т.е. получить весовые коэффициенты.

import math

# REX – массив отсчётов исходной ИХ
REX = [0.0] * 1024
# T – буфер временного хранения данных
T = [0.0] * 1024

PI = 3.14159265
M = 40  # Параметр, определяющий порядок фильтра (41 порядок)

# Заглушка для подпрограммы загрузки отсчётов ИХ
def load_impulse_response():
    # В реальной программе здесь должен быть код загрузки данных
    # Например, чтение из файла или расчёт значений
    pass

# Вызов функции загрузки ИХ
load_impulse_response()

# Сдвиг (циклический) ИХ на M/2 отсчётов вправо
for i in range(1024):
    index = i + M // 2
    if index > 1023:
        index = index - 1024
    T[index] = REX[i]

# Копирование данных из буфера T обратно в REX
for i in range(1024):
    REX[i] = T[i]

# Усечение ИХ и формирование оконной функцией
for i in range(1024):
    if i <= M:
        # Применение оконной функции Хэмминга
        REX[i] = REX[i] * (0.54 - 0.46 * math.cos(2 * PI * i / M))
    else:
        REX[i] = 0

# Результирующая ИХ размещается в REX[0]…REX[40]
# Далее можно использовать первые 41 элемент массива REX как коэффициенты фильтра

In [1]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.io import output_notebook

# Активируем вывод в Jupyter Notebook
output_notebook()

def special_filter_design(original_impulse_response=None, M=40, window_type='hamming'):
    """
    ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА
    Преобразует 1024 отсчёта исходной импульсной характеристики (ИХ)
    в (M + 1) отсчёт результирующей ИХ, т.е. получает весовые коэффициенты.
    
    Parameters:
    -----------
    original_impulse_response : numpy.array, optional
        Массив из 1024 отсчётов исходной ИХ. Если None, генерируется тестовая ИХ.
    M : int, optional
        Параметр, определяющий порядок фильтра (M+1 коэффициентов), по умолчанию 40
    window_type : str, optional
        Тип оконной функции, по умолчанию 'hamming'
    
    Returns:
    --------
    final_coefficients : numpy.array
        Результирующая ИХ (весовые коэффициенты фильтра)
    """
    
    # Аналог строк 150-160: объявление массивов
    # REX[ ] – массив отсчётов исходной ИХ (1024 элемента)
    # T[ ] – буфер временного хранения данных (1024 элемента)
    REX = np.zeros(1024)
    T = np.zeros(1024)
    
    # Аналог строки 180: определение константы π
    PI = np.pi
    
    # Аналог строки 190: параметр, определяющий порядок фильтра
    # M = 40 соответствует фильтру 41-го порядка
    
    # Аналог строки 210: загрузка исходной импульсной характеристики
    # Если исходная ИХ не предоставлена, создаем тестовую ИХ
    if original_impulse_response is None:
        # Создаем тестовую импульсную характеристику (например, прямоугольный импульс)
        REX = generate_test_impulse_response(1024)
    else:
        if len(original_impulse_response) != 1024:
            raise ValueError("Исходная импульсная характеристика должна содержать 1024 отсчета")
        REX = original_impulse_response.copy()
    
    # Сохраняем исходную ИХ для визуализации
    original_REX = REX.copy()
    
    # Аналог строк 230-270: циклический сдвиг ИХ на M/2 отсчётов вправо
    # В BASIC: FOR I% = 0 TO 1023
    for i in range(1024):
        # Вычисляем новый индекс с циклическим сдвигом
        # В BASIC: INDEX% = I% + M%/2
        index = i + M // 2
        
        # Обеспечиваем циклический сдвиг (индекс по модулю 1024)
        # В BASIC: IF INDEX% > 1023 THEN INDEX% = INDEX%-1024
        if index > 1023:
            index = index - 1024
        
        # Выполняем сдвиг
        # В BASIC: T[INDEX%] = REX[I%]
        T[index] = REX[i]
    
    # Копируем результат сдвига обратно в REX
    # Аналог строк 290-310: FOR I% = 0 TO 1023: REX[I%] = T[I%]: NEXT I%
    for i in range(1024):
        REX[i] = T[i]
    
    # Сохраняем ИХ после сдвига для визуализации
    shifted_REX = REX.copy()
    
    # Аналог строк 320-360: усечение ИХ и применение оконной функции
    # В BASIC: FOR I% = 0 TO 1023
    for i in range(1024):
        # Применяем оконную функцию только к первым M+1 отсчетам
        # В BASIC: IF I% <= M% THEN REX[I%] = REX[I%] * (0.54 - 0.46 * COS(2*PI*I%/M%))
        if i <= M:
            # Применяем оконную функцию Хэмминга (как в оригинальной программе)
            if window_type == 'hamming':
                REX[i] = REX[i] * (0.54 - 0.46 * np.cos(2 * PI * i / M))
            elif window_type == 'hann':
                REX[i] = REX[i] * (0.5 - 0.5 * np.cos(2 * PI * i / M))
            elif window_type == 'blackman':
                REX[i] = REX[i] * (0.42 - 0.5 * np.cos(2 * PI * i / M) + 
                                   0.08 * np.cos(4 * PI * i / M))
            # Для прямоугольного окна не делаем ничего
        else:
            # Обнуляем отсчеты за пределами M
            # В BASIC: IF I% > M% THEN REX[I%] = 0
            REX[i] = 0
    
    # Аналог строки 370: результирующая ИХ размещается в REX[0]…REX[M]
    # Извлекаем только первые M+1 коэффициентов как финальный результат
    final_coefficients = REX[:M+1].copy()
    
    # Нормируем коэффициенты для единичного усиления на нулевой частоте
    final_coefficients = final_coefficients / np.sum(final_coefficients)
    
    return final_coefficients, original_REX, shifted_REX

def generate_test_impulse_response(length):
    """
    Генерация тестовой импульсной характеристики для демонстрации
    
    Parameters:
    -----------
    length : int
        Длина импульсной характеристики
        
    Returns:
    --------
    impulse_response : numpy.array
        Тестовая импульсная характеристика
    """
    # Создаем массив нулей
    impulse_response = np.zeros(length)
    
    # Вариант 1: Прямоугольный импульс (простой вариант)
    # impulse_response[400:600] = 1.0
    
    # Вариант 2: Импульсная характеристика идеального НЧ-фильтра
    # Более реалистичный пример
    M = length // 2
    for i in range(length):
        if i == M:
            # Особый случай в центре
            impulse_response[i] = 1.0
        else:
            # Форма sinc-функции
            impulse_response[i] = np.sin(2 * np.pi * 0.1 * (i - M)) / (i - M)
    
    # Нормируем
    impulse_response = impulse_response / np.max(np.abs(impulse_response))
    
    return impulse_response

def create_special_filter_plots(original_impulse, shifted_impulse, final_coefficients, M):
    """
    Создание графиков для визуализации процесса проектирования специального фильтра
    
    Parameters:
    -----------
    original_impulse : numpy.array
        Исходная импульсная характеристика
    shifted_impulse : numpy.array
        Импульсная характеристика после циклического сдвига
    final_coefficients : numpy.array
        Финальные коэффициенты фильтра после усечения и оконной функции
    M : int
        Порядок фильтра
    """
    
    # Создаем оси для графиков
    x_original = np.arange(len(original_impulse))
    x_shifted = np.arange(len(shifted_impulse))
    x_final = np.arange(len(final_coefficients))
    
    # 1. График исходной импульсной характеристики
    p1 = figure(
        title="Исходная импульсная характеристика (1024 отсчета)",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    p1.line(x_original, original_impulse, line_color='blue', line_width=2)
    p1.circle(x_original, original_impulse, size=3, color='blue', alpha=0.5)
    
    # 2. График импульсной характеристики после циклического сдвига
    p2 = figure(
        title="ИХ после циклического сдвига на M/2 отсчетов",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    p2.line(x_shifted, shifted_impulse, line_color='green', line_width=2)
    p2.circle(x_shifted, shifted_impulse, size=3, color='green', alpha=0.5)
    
    # 3. График финальных коэффициентов фильтра
    p3 = figure(
        title=f"Результирующая ИХ фильтра ({M+1} коэффициентов)",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    p3.line(x_final, final_coefficients, line_color='red', line_width=2)
    p3.circle(x_final, final_coefficients, size=5, color='red', alpha=0.7)
    
    # Вертикальные линии для обозначения границ усечения
    p3.line([M, M], [min(final_coefficients), max(final_coefficients)], 
            line_color='black', line_dash='dashed', line_width=1)
    
    # 4. График сравнения всех этапов (масштабированный)
    p4 = figure(
        title="Сравнение всех этапов обработки ИХ",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    
    # Показываем только часть исходной ИХ для наглядности
    show_points = min(200, len(original_impulse))
    p4.line(x_original[:show_points], original_impulse[:show_points], 
            legend_label='Исходная ИХ', line_color='blue', line_width=2)
    p4.line(x_shifted[:show_points], shifted_impulse[:show_points], 
            legend_label='ИХ после сдвига', line_color='green', line_width=2)
    p4.line(x_final, final_coefficients, 
            legend_label='Финальные коэффициенты', line_color='red', line_width=3)
    p4.legend.location = "top_right"
    
    # Создаем сетку графиков
    plot_grid = gridplot([[p1, p2], [p3, p4]])
    show(plot_grid)

def demo_special_filter(M=40, window_type='hamming'):
    """
    Демонстрационная функция для проектирования специального фильтра
    
    Parameters:
    -----------
    M : int, optional
        Порядок фильтра, по умолчанию 40
    window_type : str, optional
        Тип оконной функции, по умолчанию 'hamming'
    """
    
    print("=" * 60)
    print("ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА")
    print("=" * 60)
    print(f"Параметры:")
    print(f"  Порядок фильтра M: {M} (фильтр {M+1}-го порядка)")
    print(f"  Тип оконной функции: {window_type}")
    print(f"  Размер исходной ИХ: 1024 отсчета")
    print(f"  Размер результирующей ИХ: {M+1} отсчетов")
    print()
    
    # Проектируем фильтр
    final_coefficients, original_impulse, shifted_impulse = special_filter_design(
        M=M, window_type=window_type
    )
    
    # Создаем графики
    create_special_filter_plots(original_impulse, shifted_impulse, final_coefficients, M)
    
    # Выводим информацию о результате
    print("РЕЗУЛЬТАТЫ:")
    print(f"  Получено коэффициентов: {len(final_coefficients)}")
    print(f"  Сумма коэффициентов: {np.sum(final_coefficients):.6f}")
    print(f"  Максимальный коэффициент: {np.max(final_coefficients):.6f}")
    print(f"  Минимальный коэффициент: {np.min(final_coefficients):.6f}")
    print()
    print("Первые 10 коэффициентов:")
    for i in range(min(10, len(final_coefficients))):
        print(f"    h[{i}] = {final_coefficients[i]:.6f}")
    
    return final_coefficients, original_impulse, shifted_impulse

# Дополнительные функции для анализа

def analyze_frequency_response(coefficients, sampling_rate=1.0):
    """
    Анализ частотной характеристики полученного фильтра
    
    Parameters:
    -----------
    coefficients : numpy.array
        Коэффициенты фильтра
    sampling_rate : float, optional
        Частота дискретизации, по умолчанию 1.0
    """
    from scipy import signal
    
    # Рассчитываем АЧХ
    w, h = signal.freqz(coefficients, worN=1024)
    frequencies = w * sampling_rate / (2 * np.pi)
    magnitude = np.abs(h)
    
    # Создаем график АЧХ
    p = figure(
        title="Амплитудно-частотная характеристика (АЧХ) фильтра",
        width=600, height=300,
        x_axis_label='Частота (нормированная)',
        y_axis_label='Коэффициент передачи'
    )
    p.line(frequencies, magnitude, line_color='purple', line_width=2)
    
    show(p)
    
    return frequencies, magnitude

# Запуск демонстрации
if __name__ == "__main__":
    # Демонстрация с параметрами по умолчанию
    print("Демонстрация программы расчета специального фильтра...")
    coefficients, original, shifted = demo_special_filter()
    
    # Дополнительный анализ АЧХ
    print("\n" + "=" * 60)
    print("АНАЛИЗ ЧАСТОТНОЙ ХАРАКТЕРИСТИКИ")
    print("=" * 60)
    freq, mag = analyze_frequency_response(coefficients)
    
    # Демонстрация с другими параметрами
    print("\n" + "=" * 60)
    print("ДЕМОНСТРАЦИЯ С ДРУГИМИ ПАРАМЕТРАМИ")
    print("=" * 60)
    coefficients2, original2, shifted2 = demo_special_filter(M=60, window_type='blackman')
    analyze_frequency_response(coefficients2)

Loading BokehJS ...

Демонстрация программы расчета специального фильтра...
ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА
Параметры:
  Порядок фильтра M: 40 (фильтр 41-го порядка)
  Тип оконной функции: hamming
  Размер исходной ИХ: 1024 отсчета
  Размер результирующей ИХ: 41 отсчетов



РЕЗУЛЬТАТЫ:
  Получено коэффициентов: 41
  Сумма коэффициентов: 1.000000
  Максимальный коэффициент: 1.010839
  Минимальный коэффициент: -1.099539

Первые 10 коэффициентов:
    h[0] = -0.091539
    h[1] = -0.097820
    h[2] = -0.072202
    h[3] = 0.000000
    h[4] = 0.117744
    h[5] = 0.243231
    h[6] = 0.304790
    h[7] = 0.230906
    h[8] = -0.000000
    h[9] = -0.325041

АНАЛИЗ ЧАСТОТНОЙ ХАРАКТЕРИСТИКИ



ДЕМОНСТРАЦИЯ С ДРУГИМИ ПАРАМЕТРАМИ
ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА
Параметры:
  Порядок фильтра M: 60 (фильтр 61-го порядка)
  Тип оконной функции: blackman
  Размер исходной ИХ: 1024 отсчета
  Размер результирующей ИХ: 61 отсчетов



РЕЗУЛЬТАТЫ:
  Получено коэффициентов: 61
  Сумма коэффициентов: 1.000000
  Максимальный коэффициент: 0.900797
  Минимальный коэффициент: -0.960719

Первые 10 коэффициентов:
    h[0] = 0.000000
    h[1] = -0.001009
    h[2] = -0.002519
    h[3] = -0.000000
    h[4] = 0.010482
    h[5] = 0.027258
    h[6] = 0.040533
    h[7] = 0.035305
    h[8] = 0.000000
    h[9] = -0.062773


In [9]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.io import output_notebook

# Активируем вывод в Jupyter Notebook
output_notebook()

# Конфигурационные параметры (замена констант)
DEFAULT_IR_LENGTH = 16  # Длина исходной импульсной характеристики
DEFAULT_FILTER_ORDER = 8 # Порядок фильтра по умолчанию

def special_filter_design(original_impulse_response=None, M=DEFAULT_FILTER_ORDER, 
                         window_type='hamming', N=DEFAULT_IR_LENGTH):
    """
    ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА
    Преобразует N отсчётов исходной импульсной характеристики (ИХ)
    в (M + 1) отсчёт результирующей ИХ, т.е. получает весовые коэффициенты.
    
    Parameters:
    -----------
    original_impulse_response : numpy.array, optional
        Массив из N отсчётов исходной ИХ. Если None, генерируется тестовая ИХ.
    M : int, optional
        Параметр, определяющий порядок фильтра (M+1 коэффициентов), по умолчанию 40
    window_type : str, optional
        Тип оконной функции, по умолчанию 'hamming'
    N : int, optional
        Длина исходной импульсной характеристики, по умолчанию 1024
    
    Returns:
    --------
    final_coefficients : numpy.array
        Результирующая ИХ (весовые коэффициенты фильтра)
    original_REX : numpy.array
        Исходная импульсная характеристика
    shifted_REX : numpy.array
        Импульсная характеристика после циклического сдвига
    """
    
    # Проверка корректности параметров
    if M >= N:
        raise ValueError(f"Порядок фильтра M={M} должен быть меньше длины ИХ N={N}")
    
    # Аналог строк 150-160: объявление массивов
    # REX[ ] – массив отсчётов исходной ИХ (N элементов)
    # T[ ] – буфер временного хранения данных (N элементов)
    REX = np.zeros(N)
    T = np.zeros(N)
    
    # Аналог строки 180: определение константы π
    PI = np.pi
    
    # Аналог строки 190: параметр, определяющий порядок фильтра
    # M = 40 соответствует фильтру 41-го порядка
    
    # Аналог строки 210: загрузка исходной импульсной характеристики
    # Если исходная ИХ не предоставлена, создаем тестовую ИХ
    if original_impulse_response is None:
        # Создаем тестовую импульсную характеристику
        REX = generate_test_impulse_response(N)
    else:
        if len(original_impulse_response) != N:
            raise ValueError(f"Исходная импульсная характеристика должна содержать {N} отсчетов")
        REX = original_impulse_response.copy()
    
    # Сохраняем исходную ИХ для визуализации
    original_REX = REX.copy()
    
    # Аналог строк 230-270: циклический сдвиг ИХ на M/2 отсчётов вправо
    # В BASIC: FOR I% = 0 TO N-1
    print("REX 1 ---------------------------------------------------------------------------------")
    print(REX)
    print("index 1 -------------------------------------------------------------------------------")
    for i in range(N):
        # Вычисляем новый индекс с циклическим сдвигом
        # В BASIC: INDEX% = I% + M%/2
        index = i + M // 2
        
        # Обеспечиваем циклический сдвиг (индекс по модулю N)
        # В BASIC: IF INDEX% > N-1 THEN INDEX% = INDEX%-N
        if index > N - 1:
            index = index - N
        
        # Выполняем сдвиг
        # В BASIC: T[INDEX%] = REX[I%]
        T[index] = REX[i]

        print(index)
    # Копируем результат сдвига обратно в REX
    # Аналог строк 290-310: FOR I% = 0 TO N-1: REX[I%] = T[I%]: NEXT I%
    print("T 1 ------------------------------------------------------------------------------------")
    print(T)
    for i in range(N):
        REX[i] = T[i]
    print("shifted_REX ----------------------------------------------------------------------------")
    print(REX)
    # Сохраняем ИХ после сдвига для визуализации
    shifted_REX = REX.copy()
    
    # Аналог строк 320-360: усечение ИХ и применение оконной функции
    # В BASIC: FOR I% = 0 TO N-1
    for i in range(N):
        # Применяем оконную функцию только к первым M+1 отсчетам
        # В BASIC: IF I% <= M% THEN REX[I%] = REX[I%] * (0.54 - 0.46 * COS(2*PI*I%/M%))
        if i <= M:
            # Применяем оконную функцию Хэмминга (как в оригинальной программе)
            if window_type == 'hamming':
                REX[i] = REX[i] * (0.54 - 0.46 * np.cos(2 * PI * i / M))
            elif window_type == 'hann':
                REX[i] = REX[i] * (0.5 - 0.5 * np.cos(2 * PI * i / M))
            elif window_type == 'blackman':
                REX[i] = REX[i] * (0.42 - 0.5 * np.cos(2 * PI * i / M) + 
                                   0.08 * np.cos(4 * PI * i / M))
            # Для прямоугольного окна не делаем ничего
        else:
            # Обнуляем отсчеты за пределами M
            # В BASIC: IF I% > M% THEN REX[I%] = 0
            REX[i] = 0
    
    # Аналог строки 370: результирующая ИХ размещается в REX[0]…REX[M]
    # Извлекаем только первые M+1 коэффициентов как финальный результат
    final_coefficients = REX[:M+1].copy()
    
    # Нормируем коэффициенты для единичного усиления на нулевой частоте
    final_coefficients = final_coefficients / np.sum(final_coefficients)
    
    return final_coefficients, original_REX, shifted_REX

def generate_test_impulse_response(length):
    """
    Генерация тестовой импульсной характеристики для демонстрации
    
    Parameters:
    -----------
    length : int
        Длина импульсной характеристики
        
    Returns:
    --------
    impulse_response : numpy.array
        Тестовая импульсная характеристика
    """
    # Создаем массив нулей
    impulse_response = np.zeros(length)
    
    # Вариант 2: Импульсная характеристика идеального НЧ-фильтра
    # Более реалистичный пример
    M = length // 2
    for i in range(length):
        if i == M:
            # Особый случай в центре
            impulse_response[i] = 1.0
        else:
            # Форма sinc-функции
            impulse_response[i] = np.sin(2 * np.pi * 0.1 * (i - M)) / (i - M)
    
    # Нормируем
    impulse_response = impulse_response / np.max(np.abs(impulse_response))
    
    return impulse_response

def create_special_filter_plots(original_impulse, shifted_impulse, final_coefficients, M, N):
    """
    Создание графиков для визуализации процесса проектирования специального фильтра
    
    Parameters:
    -----------
    original_impulse : numpy.array
        Исходная импульсная характеристика
    shifted_impulse : numpy.array
        Импульсная характеристика после циклического сдвига
    final_coefficients : numpy.array
        Финальные коэффициенты фильтра после усечения и оконной функции
    M : int
        Порядок фильтра
    N : int
        Длина исходной импульсной характеристики
    """
    
    # Создаем оси для графиков
    x_original = np.arange(len(original_impulse))
    x_shifted = np.arange(len(shifted_impulse))
    x_final = np.arange(len(final_coefficients))
    
    # 1. График исходной импульсной характеристики
    p1 = figure(
        title=f"Исходная импульсная характеристика ({N} отсчетов)",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    p1.line(x_original, original_impulse, line_color='blue', line_width=2)
    p1.circle(x_original, original_impulse, size=3, color='blue', alpha=0.5)
    
    # 2. График импульсной характеристики после циклического сдвига
    p2 = figure(
        title=f"ИХ после циклического сдвига на M/2={M//2} отсчетов",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    p2.line(x_shifted, shifted_impulse, line_color='green', line_width=2)
    p2.circle(x_shifted, shifted_impulse, size=3, color='green', alpha=0.5)
    
    # 3. График финальных коэффициентов фильтра
    p3 = figure(
        title=f"Результирующая ИХ фильтра ({M+1} коэффициентов)",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    p3.line(x_final, final_coefficients, line_color='red', line_width=2)
    p3.circle(x_final, final_coefficients, size=5, color='red', alpha=0.7)
    
    # Вертикальные линии для обозначения границ усечения
    p3.line([M, M], [min(final_coefficients), max(final_coefficients)], 
            line_color='black', line_dash='dashed', line_width=1)
    
    # 4. График сравнения всех этапов (масштабированный)
    p4 = figure(
        title="Сравнение всех этапов обработки ИХ",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    
    # Показываем только часть исходной ИХ для наглядности
    show_points = min(200, len(original_impulse))
    p4.line(x_original[:show_points], original_impulse[:show_points], 
            legend_label='Исходная ИХ', line_color='blue', line_width=2)
    p4.line(x_shifted[:show_points], shifted_impulse[:show_points], 
            legend_label='ИХ после сдвига', line_color='green', line_width=2)
    p4.line(x_final, final_coefficients, 
            legend_label='Финальные коэффициенты', line_color='red', line_width=3)
    p4.legend.location = "top_right"
    
    # Создаем сетку графиков
    plot_grid = gridplot([[p1, p2], [p3, p4]])
    show(plot_grid)

def demo_special_filter(M=DEFAULT_FILTER_ORDER, window_type='hamming', N=DEFAULT_IR_LENGTH):
    """
    Демонстрационная функция для проектирования специального фильтра
    
    Parameters:
    -----------
    M : int, optional
        Порядок фильтра, по умолчанию 40
    window_type : str, optional
        Тип оконной функции, по умолчанию 'hamming'
    N : int, optional
        Длина исходной импульсной характеристики, по умолчанию 1024
    """
    
    print("=" * 60)
    print("ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА")
    print("=" * 60)
    print(f"Параметры:")
    print(f"  Порядок фильтра M: {M} (фильтр {M+1}-го порядка)")
    print(f"  Тип оконной функции: {window_type}")
    print(f"  Размер исходной ИХ (N): {N} отсчетов")
    print(f"  Размер результирующей ИХ: {M+1} отсчетов")
    print(f"  Циклический сдвиг: {M//2} отсчетов")
    print()
    
    # Проектируем фильтр
    final_coefficients, original_impulse, shifted_impulse = special_filter_design(
        M=M, window_type=window_type, N=N
    )
    
    # Создаем графики
    create_special_filter_plots(original_impulse, shifted_impulse, final_coefficients, M, N)
    
    # Выводим информацию о результате
    print("РЕЗУЛЬТАТЫ:")
    print(f"  Получено коэффициентов: {len(final_coefficients)}")
    print(f"  Сумма коэффициентов: {np.sum(final_coefficients):.6f}")
    print(f"  Максимальный коэффициент: {np.max(final_coefficients):.6f}")
    print(f"  Минимальный коэффициент: {np.min(final_coefficients):.6f}")
    print()
    print("Первые 10 коэффициентов:")
    for i in range(min(10, len(final_coefficients))):
        print(f"    h[{i}] = {final_coefficients[i]:.6f}")
    
    return final_coefficients, original_impulse, shifted_impulse

def analyze_frequency_response(coefficients, sampling_rate=1.0):
    """
    Анализ частотной характеристики полученного фильтра
    
    Parameters:
    -----------
    coefficients : numpy.array
        Коэффициенты фильтра
    sampling_rate : float, optional
        Частота дискретизации, по умолчанию 1.0
    """
    from scipy import signal
    
    # Рассчитываем АЧХ
    w, h = signal.freqz(coefficients, worN=1024)
    frequencies = w * sampling_rate / (2 * np.pi)
    magnitude = np.abs(h)
    
    # Создаем график АЧХ
    p = figure(
        title="Амплитудно-частотная характеристика (АЧХ) фильтра",
        width=600, height=300,
        x_axis_label='Частота (нормированная)',
        y_axis_label='Коэффициент передачи'
    )
    p.line(frequencies, magnitude, line_color='purple', line_width=2)
    
    show(p)
    
    return frequencies, magnitude

# Функции для демонстрации различных конфигураций
def demo_different_configurations():
    """
    Демонстрация работы фильтра с различными конфигурациями параметров
    """
    configurations = [
        {"M": 20, "N": 512, "window_type": "hamming", "description": "Короткий фильтр"},
        {"M": 40, "N": 1024, "window_type": "hamming", "description": "Стандартный фильтр"},
        {"M": 60, "N": 2048, "window_type": "blackman", "description": "Длинный фильтр с окном Блэкмана"},
        {"M": 80, "N": 4096, "window_type": "hann", "description": "Очень длинный фильтр"},
    ]
    
    for config in configurations:
        print(f"\n{'='*60}")
        print(f"КОНФИГУРАЦИЯ: {config['description']}")
        print(f"{'='*60}")
        
        coefficients, original, shifted = demo_special_filter(
            M=config["M"],
            window_type=config["window_type"],
            N=config["N"]
        )
        
        # Краткий анализ АЧХ
        freq, mag = analyze_frequency_response(coefficients)
        
        # Находим частоту среза (-3 дБ)
        cutoff_index = np.where(mag < 0.707)[0]
        if len(cutoff_index) > 0:
            cutoff_freq = freq[cutoff_index[0]]
            print(f"  Частота среза (-3 дБ): {cutoff_freq:.4f}")
        
        print()

# Запуск демонстрации
if __name__ == "__main__":
    # Демонстрация с параметрами по умолчанию
    print("Демонстрация программы расчета специального фильтра...")
    coefficients, original, shifted = demo_special_filter()
    
    # Дополнительный анализ АЧХ
    print("\n" + "=" * 60)
    print("АНАЛИЗ ЧАСТОТНОЙ ХАРАКТЕРИСТИКИ")
    print("=" * 60)
    freq, mag = analyze_frequency_response(coefficients)
    
    # Демонстрация с другими параметрами
    print("\n" + "=" * 60)
    print("ДЕМОНСТРАЦИЯ С ДРУГИМИ ПАРАМЕТРАМИ")
    print("=" * 60)
    coefficients2, original2, shifted2 = demo_special_filter(M=60, window_type='blackman', N=2048)
    analyze_frequency_response(coefficients2)
    
    # Демонстрация различных конфигураций
    print("\n" + "=" * 60)
    print("СРАВНЕНИЕ РАЗЛИЧНЫХ КОНФИГУРАЦИЙ")
    print("=" * 60)
    # demo_different_configurations()

Loading BokehJS ...

Демонстрация программы расчета специального фильтра...
ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА
Параметры:
  Порядок фильтра M: 8 (фильтр 9-го порядка)
  Тип оконной функции: hamming
  Размер исходной ИХ (N): 16 отсчетов
  Размер результирующей ИХ: 9 отсчетов
  Циклический сдвиг: 4 отсчетов

REX 1 ---------------------------------------------------------------------------------
[-1.18882065e-01 -1.35865217e-01 -9.79642087e-02  2.44929360e-17
  1.46946313e-01  3.17018839e-01  4.75528258e-01  5.87785252e-01
  1.00000000e+00  5.87785252e-01  4.75528258e-01  3.17018839e-01
  1.46946313e-01  2.44929360e-17 -9.79642087e-02 -1.35865217e-01]
index 1 -------------------------------------------------------------------------------
4
5
6
7
8
9
10
11
12
13
14
15
0
1
2
3
T 1 ------------------------------------------------------------------------------------
[ 1.46946313e-01  2.44929360e-17 -9.79642087e-02 -1.35865217e-01
 -1.18882065e-01 -1.35865217e-01 -9.79642087e-02  2.44929360e-17
  1.46946313e-0

РЕЗУЛЬТАТЫ:
  Получено коэффициентов: 9
  Сумма коэффициентов: 1.000000
  Максимальный коэффициент: 0.272483
  Минимальный коэффициент: -0.026945

Первые 10 коэффициентов:
    h[0] = -0.026945
    h[1] = -0.000000
    h[2] = 0.121251
    h[3] = 0.269453
    h[4] = 0.272483
    h[5] = 0.269453
    h[6] = 0.121251
    h[7] = -0.000000
    h[8] = -0.026945

АНАЛИЗ ЧАСТОТНОЙ ХАРАКТЕРИСТИКИ



ДЕМОНСТРАЦИЯ С ДРУГИМИ ПАРАМЕТРАМИ
ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА
Параметры:
  Порядок фильтра M: 60 (фильтр 61-го порядка)
  Тип оконной функции: blackman
  Размер исходной ИХ (N): 2048 отсчетов
  Размер результирующей ИХ: 61 отсчетов
  Циклический сдвиг: 30 отсчетов

REX 1 ---------------------------------------------------------------------------------
[0.00057401 0.00092967 0.00093058 ... 0.0005757  0.00093058 0.00092967]
index 1 -------------------------------------------------------------------------------
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165

РЕЗУЛЬТАТЫ:
  Получено коэффициентов: 61
  Сумма коэффициентов: 1.000000
  Максимальный коэффициент: 0.370131
  Минимальный коэффициент: -0.317294

Первые 10 коэффициентов:
    h[0] = -0.000000
    h[1] = -0.000000
    h[2] = -0.000946
    h[3] = -0.003507
    h[4] = -0.006387
    h[5] = -0.006350
    h[6] = 0.000000
    h[7] = 0.013336
    h[8] = 0.029256
    h[9] = 0.038447



СРАВНЕНИЕ РАЗЛИЧНЫХ КОНФИГУРАЦИЙ


In [10]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.io import output_notebook

# Активируем вывод в Jupyter Notebook
output_notebook()

# Конфигурационные параметры
DEFAULT_FREQ_POINTS = 513  # Количество точек в АЧХ (0 до Fs/2)
DEFAULT_IR_LENGTH = 1024   # Длина импульсной характеристики после обратного БПФ
DEFAULT_FILTER_ORDER = 40  # Порядок фильтра по умолчанию

def design_special_filter(desired_magnitude=None, desired_phase=None, M=DEFAULT_FILTER_ORDER, 
                         window_type='hamming', N_fft=DEFAULT_IR_LENGTH):
    """
    ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА ПО ЗАДАННОЙ ЧАСТОТНОЙ ХАРАКТЕРИСТИКЕ
    
    Parameters:
    -----------
    desired_magnitude : numpy.array, optional
        Массив из 513 отсчётов желаемой АЧХ (от 0 до Fs/2)
    desired_phase : numpy.array, optional
        Массив из 513 отсчётов желаемой ФЧХ в радианах. Если None - нулевая фаза.
    M : int, optional
        Порядок фильтра (M+1 коэффициентов)
    window_type : str, optional
        Тип оконной функции
    N_fft : int, optional
        Длина БПФ (по умолчанию 1024)
    
    Returns:
    --------
    final_coefficients : numpy.array
        Результирующая ИХ фильтра (M+1 коэффициентов)
    original_impulse : numpy.array
        Исходная импульсная характеристика после обратного БПФ
    shifted_impulse : numpy.array
        ИХ после циклического сдвига
    computed_freq_response : tuple
        Рассчитанная АЧХ финального фильтра (частоты, амплитуда)
    """
    
    # Шаг 1: Задание желаемой частотной характеристики
    if desired_magnitude is None:
        # Создаем тестовую АЧХ (идеальный НЧ-фильтр)
        desired_magnitude = generate_test_frequency_response(DEFAULT_FREQ_POINTS)
    
    if desired_phase is None:
        # Нулевая фаза по умолчанию
        desired_phase = np.zeros(DEFAULT_FREQ_POINTS)
    
    # Проверка размеров
    if len(desired_magnitude) != DEFAULT_FREQ_POINTS:
        raise ValueError(f"Желаемая АЧХ должна содержать {DEFAULT_FREQ_POINTS} отсчетов")
    
    # Сохраняем желаемую АЧХ для визуализации
    desired_freq = np.linspace(0, 0.5, DEFAULT_FREQ_POINTS)  # Нормированные частоты 0...0.5
    
    # Шаг 2: Преобразование в прямоугольные координаты и построение полного спектра
    complex_freq_response = desired_magnitude * np.exp(1j * desired_phase)
    
    # Создаем полный спектр для обратного БПФ (1024 точки)
    full_spectrum = build_full_spectrum(complex_freq_response, N_fft)
    
    # Шаг 3: Обратное БПФ для перехода во временную область
    original_impulse = np.fft.ifft(full_spectrum).real
    # Нормировка (БПФ в numpy не нормирован)
    original_impulse = original_impulse / len(original_impulse)
    
    # Шаг 4: Формирование импульсной характеристики (сдвиг, усечение, оконная функция)
    final_coefficients, shifted_impulse = shape_impulse_response(
        original_impulse, M, window_type
    )
    
    # Шаг 5: Проверка - расчет АЧХ полученного фильтра с дополнением нулями
    computed_freq = compute_frequency_response(final_coefficients, N_fft)
    
    return final_coefficients, original_impulse, shifted_impulse, (desired_freq, desired_magnitude), computed_freq

def build_full_spectrum(half_spectrum, N_fft):
    """
    Построение полного спектра из половинного спектра (0...0.5)
    для обратного БПФ с учетом симметрии для вещественного сигнала
    """
    full_spectrum = np.zeros(N_fft, dtype=complex)
    N_half = len(half_spectrum)
    
    # Первая половина (положительные частоты)
    full_spectrum[:N_half] = half_spectrum
    
    # Вторая половина (отрицательные частоты) - комплексно-сопряженная симметрия
    # Для вещественного сигнала: H[k] = H[N-k]*
    for i in range(1, N_half - 1):
        full_spectrum[N_fft - i] = np.conj(half_spectrum[i])
    
    return full_spectrum

def shape_impulse_response(impulse_response, M, window_type):
    """
    Формирование импульсной характеристики: сдвиг, усечение, оконное взвешивание
    """
    N = len(impulse_response)
    T = np.zeros(N)
    
    # Циклический сдвиг на M/2 отсчетов вправо
    for i in range(N):
        index = i + M // 2
        if index >= N:
            index = index - N
        T[index] = impulse_response[i]
    
    # Копируем результат сдвига
    shifted_impulse = T.copy()
    
    # Усечение и применение оконной функции
    for i in range(N):
        if i <= M:
            # Применяем оконную функцию
            if window_type == 'hamming':
                T[i] = T[i] * (0.54 - 0.46 * np.cos(2 * np.pi * i / M))
            elif window_type == 'hann':
                T[i] = T[i] * (0.5 - 0.5 * np.cos(2 * np.pi * i / M))
            elif window_type == 'blackman':
                T[i] = T[i] * (0.42 - 0.5 * np.cos(2 * np.pi * i / M) + 
                               0.08 * np.cos(4 * np.pi * i / M))
        else:
            # Обнуляем отсчеты за пределами M
            T[i] = 0
    
    # Извлекаем финальные коэффициенты
    final_coefficients = T[:M+1].copy()
    
    # Нормировка для единичного усиления на нулевой частоте
    final_coefficients = final_coefficients / np.sum(final_coefficients)
    
    return final_coefficients, shifted_impulse

def compute_frequency_response(coefficients, N_fft):
    """
    Расчет АЧХ фильтра с дополнением нулями для повышения разрешения
    """
    # Дополняем импульсную характеристику нулями
    padded_ir = np.zeros(N_fft)
    padded_ir[:len(coefficients)] = coefficients
    
    # Прямое БПФ для получения частотной характеристики
    freq_response = np.fft.fft(padded_ir)
    
    # Берем только первую половину (положительные частоты)
    N_half = N_fft // 2 + 1
    frequencies = np.linspace(0, 0.5, N_half)  # Нормированные частоты 0...0.5
    magnitude = np.abs(freq_response[:N_half])
    
    return frequencies, magnitude

def generate_test_frequency_response(N_points):
    """
    Генерация тестовой АЧХ (идеальный НЧ-фильтр с резким срезом)
    """
    magnitude = np.ones(N_points)
    cutoff_index = N_points // 3  # Частота среза на 1/3 от Fs/2
    
    # Резкий срез - как в книге
    magnitude[cutoff_index:] = 0
    
    # Можно добавить другие формы АЧХ для тестирования:
    # - Полосовой фильтр
    # - Режекторный фильтр  
    # - АЧХ с подъемами и спадами
    
    return magnitude

def create_comprehensive_plots(desired_freq_response, original_impulse, shifted_impulse, 
                              final_coefficients, computed_freq_response, M, N_fft):
    """
    Создание комплексных графиков для анализа процесса проектирования фильтра
    """
    
    desired_freq, desired_magnitude = desired_freq_response
    computed_freq, computed_magnitude = computed_freq_response
    
    # 1. График желаемой и полученной АЧХ
    p1 = figure(
        title="Желаемая и полученная АЧХ",
        width=600, height=300,
        x_axis_label='Нормированная частота',
        y_axis_label='Амплитуда'
    )
    p1.line(desired_freq, desired_magnitude, 
            legend_label='Желаемая АЧХ', line_color='blue', line_width=2)
    p1.line(computed_freq, computed_magnitude, 
            legend_label='Полученная АЧХ', line_color='red', line_width=2)
    p1.legend.location = "top_right"
    
    # 2. График исходной импульсной характеристики (после обратного БПФ)
    p2 = figure(
        title=f"Исходная ИХ после обратного БПФ ({N_fft} отсчетов)",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    x_original = np.arange(len(original_impulse))
    p2.line(x_original, original_impulse, line_color='green', line_width=2)
    
    # 3. График импульсной характеристики после циклического сдвига
    p3 = figure(
        title=f"ИХ после циклического сдвига на M/2={M//2} отсчетов",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    x_shifted = np.arange(len(shifted_impulse))
    p3.line(x_shifted, shifted_impulse, line_color='orange', line_width=2)
    
    # 4. График финальных коэффициентов фильтра
    p4 = figure(
        title=f"Результирующая ИХ фильтра ({M+1} коэффициентов)",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    x_final = np.arange(len(final_coefficients))
    p4.line(x_final, final_coefficients, line_color='red', line_width=2)
    p4.circle(x_final, final_coefficients, size=5, color='red', alpha=0.7)
    
    # 5. График сравнения всех этапов обработки ИХ
    p5 = figure(
        title="Сравнение этапов обработки ИХ (масштабированный вид)",
        width=600, height=300,
        x_axis_label='Отсчеты',
        y_axis_label='Амплитуда'
    )
    show_points = min(100, len(original_impulse))
    p5.line(x_original[:show_points], original_impulse[:show_points], 
            legend_label='Исходная ИХ', line_color='green', line_width=2)
    p5.line(x_shifted[:show_points], shifted_impulse[:show_points], 
            legend_label='ИХ после сдвига', line_color='orange', line_width=2)
    p5.line(x_final, final_coefficients, 
            legend_label='Финальные коэффициенты', line_color='red', line_width=3)
    p5.legend.location = "top_right"
    
    # 6. График ошибки аппроксимации АЧХ
    p6 = figure(
        title="Ошибка аппроксимации АЧХ",
        width=600, height=300,
        x_axis_label='Нормированная частота',
        y_axis_label='Ошибка'
    )
    
    # Интерполируем полученную АЧХ к тем же точкам, что и желаемая
    from scipy import interpolate
    interp_func = interpolate.interp1d(computed_freq, computed_magnitude, 
                                      bounds_error=False, fill_value=0)
    computed_magnitude_interp = interp_func(desired_freq)
    
    error = np.abs(desired_magnitude - computed_magnitude_interp)
    p6.line(desired_freq, error, line_color='purple', line_width=2)
    
    # Создаем сетку графиков
    plot_grid = gridplot([[p1, p2], [p3, p4], [p5, p6]])
    show(plot_grid)

def demo_special_filter_design(M=DEFAULT_FILTER_ORDER, window_type='hamming'):
    """
    Демонстрационная функция проектирования специального фильтра
    """
    
    print("=" * 70)
    print("ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА ПО ЗАДАННОЙ АЧХ")
    print("=" * 70)
    print(f"Параметры проектирования:")
    print(f"  Порядок фильтра M: {M} (фильтр {M+1}-го порядка)")
    print(f"  Тип оконной функции: {window_type}")
    print(f"  Точек в АЧХ: {DEFAULT_FREQ_POINTS}")
    print(f"  Длина БПФ: {DEFAULT_IR_LENGTH}")
    print(f"  Циклический сдвиг: {M//2} отсчетов")
    print()
    
    # Проектируем фильтр
    (final_coefficients, original_impulse, shifted_impulse, 
     desired_freq_response, computed_freq_response) = design_special_filter(
        M=M, window_type=window_type
    )
    
    # Создаем графики
    create_comprehensive_plots(
        desired_freq_response, original_impulse, shifted_impulse,
        final_coefficients, computed_freq_response, M, DEFAULT_IR_LENGTH
    )
    
    # Анализ результатов
    print("РЕЗУЛЬТАТЫ ПРОЕКТИРОВАНИЯ:")
    print(f"  Получено коэффициентов: {len(final_coefficients)}")
    print(f"  Сумма коэффициентов: {np.sum(final_coefficients):.6f}")
    
    # Оценка качества аппроксимации
    desired_freq, desired_magnitude = desired_freq_response
    computed_freq, computed_magnitude = computed_freq_response
    
    # Интерполируем для сравнения в одинаковых точках
    from scipy import interpolate
    interp_func = interpolate.interp1d(computed_freq, computed_magnitude, 
                                      bounds_error=False, fill_value=0)
    computed_magnitude_interp = interp_func(desired_freq)
    
    mse = np.mean((desired_magnitude - computed_magnitude_interp)**2)
    max_error = np.max(np.abs(desired_magnitude - computed_magnitude_interp))
    
    print(f"  СКО аппроксимации АЧХ: {mse:.6f}")
    print(f"  Максимальная ошибка: {max_error:.6f}")
    print()
    
    print("Первые 10 коэффициентов фильтра:")
    for i in range(min(10, len(final_coefficients))):
        print(f"    h[{i}] = {final_coefficients[i]:.8f}")
    
    return (final_coefficients, original_impulse, shifted_impulse, 
            desired_freq_response, computed_freq_response)

# Демонстрация влияния порядка фильтра на качество аппроксимации
def demo_filter_order_comparison():
    """
    Демонстрация влияния порядка фильтра на точность аппроксимации АЧХ
    """
    orders = [20, 40, 60, 80]
    
    print("=" * 70)
    print("СРАВНЕНИЕ ВЛИЯНИЯ ПОРЯДКА ФИЛЬТРА НА ТОЧНОСТЬ АЧХ")
    print("=" * 70)
    
    for order in orders:
        print(f"\nПорядок фильтра: {order}")
        print("-" * 40)
        
        (final_coefficients, original_impulse, shifted_impulse, 
         desired_freq_response, computed_freq_response) = design_special_filter(M=order)
        
        # Оценка качества
        desired_freq, desired_magnitude = desired_freq_response
        computed_freq, computed_magnitude = computed_freq_response
        
        from scipy import interpolate
        interp_func = interpolate.interp1d(computed_freq, computed_magnitude, 
                                          bounds_error=False, fill_value=0)
        computed_magnitude_interp = interp_func(desired_freq)
        
        mse = np.mean((desired_magnitude - computed_magnitude_interp)**2)
        max_error = np.max(np.abs(desired_magnitude - computed_magnitude_interp))
        
        print(f"  СКО: {mse:.6f}")
        print(f"  Макс. ошибка: {max_error:.6f}")
        print(f"  Коэффициентов: {len(final_coefficients)}")

# Запуск демонстрации
if __name__ == "__main__":
    # Основная демонстрация
    print("ДЕМОНСТРАЦИЯ ПРОЕКТИРОВАНИЯ СПЕЦИАЛЬНОГО ФИЛЬТРА")
    print("(в соответствии с описанием из книги)")
    print()
    
    results = demo_special_filter_design(M=40, window_type='hamming')
    
    # Демонстрация влияния порядка
    print("\n" + "=" * 70)
    demo_filter_order_comparison()

Loading BokehJS ...

ДЕМОНСТРАЦИЯ ПРОЕКТИРОВАНИЯ СПЕЦИАЛЬНОГО ФИЛЬТРА
(в соответствии с описанием из книги)

ПРОГРАММА РАСЧЁТА СПЕЦИАЛЬНОГО ФИЛЬТРА ПО ЗАДАННОЙ АЧХ
Параметры проектирования:
  Порядок фильтра M: 40 (фильтр 41-го порядка)
  Тип оконной функции: hamming
  Точек в АЧХ: 513
  Длина БПФ: 1024
  Циклический сдвиг: 20 отсчетов



РЕЗУЛЬТАТЫ ПРОЕКТИРОВАНИЯ:
  Получено коэффициентов: 41
  Сумма коэффициентов: 1.000000
  СКО аппроксимации АЧХ: 0.008440
  Максимальная ошибка: 0.490704

Первые 10 коэффициентов фильтра:
    h[0] = 0.00111397
    h[1] = 0.00122698
    h[2] = -0.00003332
    h[3] = -0.00212791
    h[4] = -0.00285979
    h[5] = 0.00006978
    h[6] = 0.00534342
    h[7] = 0.00695606
    h[8] = -0.00012928
    h[9] = -0.01178392

СРАВНЕНИЕ ВЛИЯНИЯ ПОРЯДКА ФИЛЬТРА НА ТОЧНОСТЬ АЧХ

Порядок фильтра: 20
----------------------------------------
  СКО: 0.016808
  Макс. ошибка: 0.496753
  Коэффициентов: 21

Порядок фильтра: 40
----------------------------------------
  СКО: 0.008440
  Макс. ошибка: 0.490704
  Коэффициентов: 41

Порядок фильтра: 60
----------------------------------------
  СКО: 0.005623
  Макс. ошибка: 0.484593
  Коэффициентов: 61

Порядок фильтра: 80
----------------------------------------
  СКО: 0.004217
  Макс. ошибка: 0.479203
  Коэффициентов: 81
